In [1]:
# DUOMENŲ ĮKĖLIMAS IŠ https://huggingface.co/datasets/coldstart88/triageiq-dataset

In [2]:
#!pip install datasets

In [3]:
from datasets import load_dataset
import pandas as pd

# Pagal https://pypi.org/project/datasets/:
dataset = load_dataset("coldstart88/triageiq-dataset")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'sentiment', 'urgency', 'category'],
        num_rows: 1399
    })
    validation: Dataset({
        features: ['text', 'sentiment', 'urgency', 'category'],
        num_rows: 299
    })
    test: Dataset({
        features: ['text', 'sentiment', 'urgency', 'category'],
        num_rows: 302
    })
})


In [4]:
mokymosi_duomenys = pd.DataFrame(dataset["train"])
validavimo_duomenys = pd.DataFrame(dataset["validation"])
testavimo_duomenys = pd.DataFrame(dataset["test"])

# Pagal https://pandas.pydata.org/docs/reference/api/pandas.concat.html:
visi_duomenys_apjungti = pd.concat([mokymosi_duomenys, validavimo_duomenys, testavimo_duomenys], ignore_index=True)
print(visi_duomenys_apjungti)

                                                   text sentiment urgency  \
0     this is urgent but also — whoever helped me ju...  positive    high   
1     been waiting 3 weeks for a refund and every ti...  negative  medium   
2     been waiting on this delivery longer than expe...  positive  medium   
3     do you guys do weekend deliveries or is it str...   neutral  medium   
4     my subscription renewed but i lost access at t...   neutral    high   
...                                                 ...       ...     ...   
1995  the 'forgot password' feature is literally bro...  negative     low   
1996  okay ur support is actually fire?? got my acco...  positive    high   
1997  every time i try to log in it says my password...  negative     low   
1998  whoever fixed the password reset flow recently...  positive     low   
1999  shoutout to whoever just helped me recover my ...  positive    high   

     category  
0       other  
1       other  
2       other  
3       oth

In [5]:
visi_duomenys_apjungti.to_csv("duomenys.csv", index=False)

In [7]:
# Failo nuskaitymas
visi_duomenys_apjungti = pd.read_csv("duomenys.csv")

In [22]:
# DUOMENŲ PIRMINIS APDOROJIMAS

In [8]:
# Tuščios reikšmės
visi_duomenys_apjungti.isnull().sum()

text         0
sentiment    0
urgency      0
category     0
dtype: int64

In [9]:
# Pasikartojančios reikšmės
visi_duomenys_apjungti.duplicated().sum()

np.int64(2)

In [10]:
# Pagal https://www.w3schools.com/python/pandas/ref_df_duplicated.asp tikrinama, kurios eilutės turi tokią pačią informaciją:
sudubliavusios_duomenys = visi_duomenys_apjungti.duplicated(keep=False)
print(visi_duomenys_apjungti[sudubliavusios_duomenys])

                                                   text sentiment urgency  \
491   hey @support any chance you could add a dark m...   neutral  medium   
605   omg THANK YOU for fixing that double charge so...  positive    high   
704   omg THANK YOU for fixing that double charge so...  positive    high   
1795  hey @support any chance you could add a dark m...   neutral  medium   

             category  
491   feature_request  
605           billing  
704           billing  
1795  feature_request  


In [11]:
# Pasikartojančių reikšmių šalinimas
visi_duomenys_apjungti = visi_duomenys_apjungti.drop_duplicates()

In [12]:
# Patikrinimas, ar visos pasikartojančios reikšmės pašalintos
visi_duomenys_apjungti.duplicated().sum()

np.int64(0)

In [13]:
# Patikrinama, ar klasės pasiskirsčiusios panašiai:
kategorija = visi_duomenys_apjungti["category"].value_counts()
sentimentai = visi_duomenys_apjungti["sentiment"].value_counts()
skubumas = visi_duomenys_apjungti["urgency"].value_counts()

print(kategorija)
print(sentimentai)
print(skubumas)

category
other              402
technical          400
feature_request    399
billing            399
account            398
Name: count, dtype: int64
sentiment
negative    670
positive    665
neutral     663
Name: count, dtype: int64
urgency
low       669
medium    665
high      664
Name: count, dtype: int64


In [ ]:
# SENTIMENTŲ ANALIZĖ

In [15]:
import nltk
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [16]:
# Sentimentų analizė su VADER
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

vader_duomenys = visi_duomenys_apjungti.copy()
sid = SentimentIntensityAnalyzer()
vader_duomenys["scores"] = vader_duomenys["text"].apply(lambda tekstas: sid.polarity_scores(tekstas))
vader_duomenys["compound"] = vader_duomenys["text"].apply(lambda tekstas: sid.polarity_scores(tekstas)["compound"])

def comp_score_nustatymas(compound):
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"

vader_duomenys["vader_sentiment"] = vader_duomenys["compound"].apply(comp_score_nustatymas)
visi_duomenys_apjungti["vader_sentiment"] = vader_duomenys["vader_sentiment"]
print(visi_duomenys_apjungti[["sentiment", "urgency", "category", "vader_sentiment"]])

# Kiek VADER sentimentai sutampa su jau esančiais "sentiment" stulpelyje
sutampantys_sentimentai = (visi_duomenys_apjungti["sentiment"] == visi_duomenys_apjungti["vader_sentiment"]).sum()
print("Sutampančių sentimentų skaičius:", sutampantys_sentimentai)

     sentiment urgency category vader_sentiment
0     positive    high    other        positive
1     negative  medium    other         neutral
2     positive  medium    other        positive
3      neutral  medium    other         neutral
4      neutral    high    other        negative
...        ...     ...      ...             ...
1995  negative     low  account        negative
1996  positive    high  account        positive
1997  negative     low  account        negative
1998  positive     low  account        positive
1999  positive    high  account        negative

[1998 rows x 4 columns]
Sutampančių sentimentų skaičius: 1056


In [23]:
# TEKSTO APDOROJIMAS

In [18]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
# Padalinimas į žodžius ir skyrybos ženklus (angl. tokenization)
zodzio_vienetai = []

for tekstas in visi_duomenys_apjungti["text"]:
    zodziai = nltk.word_tokenize(tekstas)
    zodzio_vienetai.append(zodziai)

visi_duomenys_apjungti["zodzio_vienetai"] = zodzio_vienetai
print(visi_duomenys_apjungti[["text", "zodzio_vienetai"]].head())

                                                text  \
0  this is urgent but also — whoever helped me ju...   
1  been waiting 3 weeks for a refund and every ti...   
2  been waiting on this delivery longer than expe...   
3  do you guys do weekend deliveries or is it str...   
4  my subscription renewed but i lost access at t...   

                                     zodzio_vienetai  
0  [this, is, urgent, but, also, —, whoever, help...  
1  [been, waiting, 3, weeks, for, a, refund, and,...  
2  [been, waiting, on, this, delivery, longer, th...  
3  [do, you, guys, do, weekend, deliveries, or, i...  
4  [my, subscription, renewed, but, i, lost, acce...  


In [29]:
# Pašalinami skyrybos ženklai, pertekliniai tarpai, skaitmenys, emoji
# Tikrinama, kokios yra žymos tekstuose
import spacy
nlp = spacy.load("en_core_web_sm")
visos_zymos = set()

for tekstas in visi_duomenys_apjungti["text"]:
    doc = nlp(tekstas)

    for token in doc:
        visos_zymos.add(token.pos_)
        '''
        if token.pos_ == "SYM":
            print(f'{token.text:{15}}')
        if token.pos_ == "X":
            print(f'{token.text:{15}}')
        '''

print("Visos rastos žymos:")
print(sorted(visos_zymos))

Visos rastos žymos:
['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']


In [36]:
# Pašalinami skaičiai, tarpai, skyrybos ženklai, simboliai, X
tekstas_be_num_space_punct_sym_x = []
for tekstas in visi_duomenys_apjungti["text"]:
    doc = nlp(tekstas)

    naujas_tekstas = " ".join(token.text for token in doc if token.pos_ not in ("NUM", "SPACE", "PUNCT", "SYM", "X"))
    tekstas_be_num_space_punct_sym_x.append(naujas_tekstas)

visi_duomenys_apjungti["tekstas_be_num_space_punct_sym_x"] = tekstas_be_num_space_punct_sym_x
print(visi_duomenys_apjungti[["text", "tekstas_be_num_space_punct_sym_x"]].head())

                                                text  \
0  this is urgent but also — whoever helped me ju...   
1  been waiting 3 weeks for a refund and every ti...   
2  been waiting on this delivery longer than expe...   
3  do you guys do weekend deliveries or is it str...   
4  my subscription renewed but i lost access at t...   

                    tekstas_be_num_space_punct_sym_x  
0  this is urgent but also whoever helped me just...  
1  been waiting weeks for a refund and every time...  
2  been waiting on this delivery longer than expe...  
3  do you guys do weekend deliveries or is it str...  
4  my subscription renewed but i lost access at t...  


In [41]:
# Nereikšmingų žodžių (angl. stopwords) šalinimas
# Tikrinama, kokie stopwords yra tekstuose
from nltk.corpus import stopwords
my_stopwords = set(stopwords.words("english"))
rasti_stopwords = set()

for tekstas in visi_duomenys_apjungti["tekstas_be_num_space_punct_sym_x"]:
    zodziai = nltk.word_tokenize(tekstas)
    for zodis in zodziai:
        if zodis.lower() in my_stopwords:
            rasti_stopwords.add(zodis.lower())

print(f"Tekstuose rasti stopwords: {sorted(rasti_stopwords)}")
print(f"Iš viso skirtingų stopwords: {len(rasti_stopwords)}")

Tekstuose rasti stopwords: ['a', 'about', 'above', 'after', 'again', 'all', 'am', 'an', 'and', 'any', 'are', 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'd', 'did', 'do', 'does', 'doing', 'down', 'during', 'each', 'few', 'for', 'from', 'had', 'has', 'have', 'having', 'he', 'here', 'how', 'i', 'if', 'in', 'into', 'is', 'it', 'its', 'itself', 'just', 'll', 'm', 'me', 'more', 'most', 'my', 'myself', 'no', 'not', 'now', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'out', 'over', 'own', 're', 's', 'same', 'she', 'should', 'so', 'some', 'such', 'than', 'that', 'the', 'their', 'them', 'then', 'there', 'these', 'they', 'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 've', 'very', 'was', 'we', 'were', 'what', 'when', 'where', 'which', 'while', 'who', 'why', 'will', 'with', 'y', 'you', 'your', 'yours', 'yourselves']
Iš viso skirtingų stopwords: 116


In [40]:
# Paliekami svarbūs stopwords
paliekami_zodziai = ["again", "but", "no", "not", "now", "more", "most", "too", "very"]
for zodis in paliekami_zodziai:
    my_stopwords.discard(zodis)

zodziai_be_stop = []

for tekstas in visi_duomenys_apjungti["tekstas_be_num_space_punct_sym_x"]:
    zodziai = nltk.word_tokenize(tekstas)
    nauji_zodziai = []
    for zodis in zodziai:
        if zodis.lower() not in my_stopwords:
            nauji_zodziai.append(zodis)

    zodziai_be_stop.append(nauji_zodziai)

visi_duomenys_apjungti["zodziai_be_stop"] = zodziai_be_stop
print(visi_duomenys_apjungti[["tekstas_be_num_space_punct_sym_x", "zodziai_be_stop"]].head())

                    tekstas_be_num_space_punct_sym_x  \
0  this is urgent but also whoever helped me just...   
1  been waiting weeks for a refund and every time...   
2  been waiting on this delivery longer than expe...   
3  do you guys do weekend deliveries or is it str...   
4  my subscription renewed but i lost access at t...   

                                     zodziai_be_stop  
0  [urgent, but, also, whoever, helped, now, dese...  
1  [waiting, weeks, refund, every, time, ask, get...  
2  [waiting, delivery, longer, expected, but, hon...  
3    [guys, weekend, deliveries, strictly, weekdays]  
4  [subscription, renewed, but, lost, access, tim...  


In [50]:
# Žodžių šaknys
from nltk.stem.porter import *
p_stemmer = PorterStemmer()

zodziu_saknys = []

for zodziai in visi_duomenys_apjungti["zodziai_be_stop"]:
    naujos_saknys = []
    for zodis in zodziai:
        zodzio_saknis = p_stemmer.stem(zodis)
        naujos_saknys.append(zodzio_saknis)

    zodziu_saknys.append(naujos_saknys)

visi_duomenys_apjungti["zodziu_saknys"] = zodziu_saknys
print(visi_duomenys_apjungti[["zodziai_be_stop", "zodziu_saknys"]].head())

                                     zodziai_be_stop  \
0  [urgent, but, also, whoever, helped, now, dese...   
1  [waiting, weeks, refund, every, time, ask, get...   
2  [waiting, delivery, longer, expected, but, hon...   
3    [guys, weekend, deliveries, strictly, weekdays]   
4  [subscription, renewed, but, lost, access, tim...   

                                       zodziu_saknys  
0  [urgent, but, also, whoever, help, now, deserv...  
1  [wait, week, refund, everi, time, ask, get, di...  
2  [wait, deliveri, longer, expect, but, honestli...  
3        [guy, weekend, deliveri, strictli, weekday]  
4  [subscript, renew, but, lost, access, time, 's...  


In [51]:
# Žodžių pagrindinės formos
zodziu_pagrindines_formos = []

for zodziai in visi_duomenys_apjungti["zodziai_be_stop"]:
    tekstas = " ".join(zodziai)
    doc = nlp(tekstas)

    naujos_pagrindines_formos = []

    for token in doc:
        pagrindine_forma = token.lemma_
        naujos_pagrindines_formos.append(pagrindine_forma)

    zodziu_pagrindines_formos.append(naujos_pagrindines_formos)

visi_duomenys_apjungti["zodziu_pagrindines_formos"] = zodziu_pagrindines_formos
print(visi_duomenys_apjungti[["zodziai_be_stop", "zodziu_pagrindines_formos"]].head(5))

                                     zodziai_be_stop  \
0  [urgent, but, also, whoever, helped, now, dese...   
1  [waiting, weeks, refund, every, time, ask, get...   
2  [waiting, delivery, longer, expected, but, hon...   
3    [guys, weekend, deliveries, strictly, weekdays]   
4  [subscription, renewed, but, lost, access, tim...   

                           zodziu_pagrindines_formos  
0  [urgent, but, also, whoever, help, now, deserv...  
1  [wait, week, refund, every, time, ask, get, di...  
2  [wait, delivery, long, expect, but, honestly, ...  
3        [guy, weekend, delivery, strictly, weekday]  
4  [subscription, renew, but, lose, access, time,...  


In [49]:
print(visi_duomenys_apjungti[["zodziai_be_stop", "zodziu_saknys", "zodziu_pagrindines_formos"]].head(10))

                                     zodziai_be_stop  \
0  [urgent, but, also, whoever, helped, now, dese...   
1  [waiting, weeks, refund, every, time, ask, get...   
2  [waiting, delivery, longer, expected, but, hon...   
3    [guys, weekend, deliveries, strictly, weekdays]   
4  [subscription, renewed, but, lost, access, tim...   
5  [kinda, odd, got, shipping, confirmation, emai...   
6  [wanted, drop, lil, appreciation, bc, ur, supp...   
7  [really, liked, guys, but, lately, every, sing...   
8  [genuinely, thought, done, today, money, line,...   
9  [telling, anyone, 'll, listen, brand, never, let]   

                                       zodziu_saknys  \
0  [urgent, but, also, whoever, help, now, deserv...   
1  [wait, week, refund, everi, time, ask, get, di...   
2  [wait, deliveri, longer, expect, but, honestli...   
3        [guy, weekend, deliveri, strictli, weekday]   
4  [subscript, renew, but, lost, access, time, 's...   
5  [kinda, odd, got, ship, confirm, email, orde

In [53]:
# Padalinimas į mokymo, validavimo ir testavimo aibes
from sklearn.model_selection import train_test_split

# Pagal https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html:
mokymo_duomenys,laikini_duomenys = train_test_split(visi_duomenys_apjungti, test_size=0.3, random_state=42, stratify=visi_duomenys_apjungti["category"])
validavimo_duomenys, testavimo_duomenys = train_test_split(laikini_duomenys, test_size=0.5, random_state=42, stratify=laikini_duomenys["category"])

print("Mokymo duomenų skaičius:", len(mokymo_duomenys))
print("Validavimo duomenų skaičius:", len(validavimo_duomenys))
print("Testavimo duomenų skaičius:", len(testavimo_duomenys))

Mokymo duomenų skaičius: 1398
Validavimo duomenų skaičius: 300
Testavimo duomenų skaičius: 300
